In [2]:
import json
import fsspec
import pandas as pd
import pyarrow.parquet as pq
from urllib.parse import urljoin
from urllib.request import urlopen
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor
pd.set_option('display.max_rows',10)

In [3]:
CATALOGURL = 'https://data.source.coop/ftw/benchmark-data/catalog.json'
COLLECTION = 'ee/ee_2024'
YEAR       = 2024
MINFIELDS  = 100
NWORKERS   = 16
REMOTE     = fsspec.filesystem('https')

In [4]:
def list_child_urls(catalogurl):
    with urlopen(catalogurl,timeout=60) as response:
        links = json.load(response)['links']
    return [urljoin(catalogurl,link['href']) for link in links if link['rel']=='child']

def build_file_url(collection,filetype):
    return urljoin(CATALOGURL,f"{collection}/{collection.split('/')[1]}_{filetype}.parquet")

def get_parquet_metadata(fileurl):
    with REMOTE.open(fileurl,block_size=2**20) as file:
        parquet = pq.ParquetFile(file)
        return parquet.schema_arrow.names,parquet.metadata.num_rows

def load_columns(fileurl,cols):
    with REMOTE.open(fileurl) as file:
        return pq.read_table(file,columns=cols).to_pandas()

def summarize_collection_labels(collection):
    fieldsurl  = build_file_url(collection,'fields')
    cols,nrows = get_parquet_metadata(fieldsurl)
    if 'hcat:code' not in cols:
        return dict(collection=collection,nfields=nrows,haslabels=False)
    counts = load_columns(fieldsurl,['hcat:code'])['hcat:code'].value_counts()
    return dict(collection=collection,
                nfields=nrows,
                haslabels=True,
                pctlabelled=100*counts.sum()/nrows,
                nclasses=len(counts),
                nkept=(counts>=MINFIELDS).sum(),
                embedmb=REMOTE.size(build_file_url(collection,'field_embeddings'))/1e6)

In [5]:
sourceurls = list_child_urls(CATALOGURL)

with ThreadPoolExecutor(NWORKERS) as executor:
    collectionurls = [url for urls in executor.map(list_child_urls,sourceurls) for url in urls]
allcollections  = ['/'.join(url.split('/')[-3:-1]) for url in collectionurls]
yearcollections = [collection for collection in allcollections if collection.endswith(str(YEAR))]

print('The FTW benchmark has...')
print(f'- {len(sourceurls)} sources (a dataset of fields from one region, like `ee`)')
print(f'- {len(allcollections)} collections (one year of one source, like `ee_2024`)')
print(f'- {len(yearcollections)} sources have {YEAR} data ({len(yearcollections)} collections)')

The FTW benchmark has...
- 37 sources (a dataset of fields from one region, like `ee`)
- 180 collections (one year of one source, like `ee_2024`)
- 23 sources have 2024 data (23 collections)


In [6]:
with ThreadPoolExecutor(NWORKERS) as executor:
    summaries = pd.DataFrame(list(executor.map(summarize_collection_labels,yearcollections))).sort_values('nfields').reset_index(drop=True)
labelled    = summaries[summaries.haslabels].drop(columns='haslabels').astype({'nclasses':int,'nkept':int}).reset_index(drop=True)
unlabelled  = summaries.loc[~summaries.haslabels,'collection']
COLLECTIONS = labelled.collection.tolist()
headers     = {'collection':'Collection','nfields':'Fields','pctlabelled':'Labelled Fields (%)','nclasses':'Crop Classes',
               'nkept':f'Crop Classes with ≥ {MINFIELDS} Fields','embedmb':'Embeddings File Size (MB)'}

print(f'Of the {len(summaries)} collections from {YEAR}...')
print(f'- {len(unlabelled)} have no HCAT crop labels')
print(f'- {len(labelled)} have HCAT crop labels, totaling {labelled.nfields.sum():,} fields and {labelled.embedmb.sum()/1e3:,.1f} GB of embeddings in total')

Of the 23 collections from 2024...
- 5 have no HCAT crop labels
- 18 have HCAT crop labels, totaling 32,552,410 fields and 29.7 GB of embeddings in total


In [7]:
display(labelled.rename(columns=headers).style.format(
    {'Fields':'{:,}','Labelled Fields (%)':'{:.1f}','Embeddings File Size (MB)':'{:,.0f}'}).hide(axis='index'))

Collection,Fields,Labelled Fields (%),Crop Classes,Crop Classes with ≥ 100 Fields,Embeddings File Size (MB)
ee/ee_2024,"166,500",100.0,123,37,154
sk/sk_2024,"275,049",98.3,128,44,255
lv/lv_2024,"424,819",100.0,105,57,394
cz/cz_2024,"426,583",100.0,160,66,395
be_vlg/be_vlg_2024,"589,749",100.0,135,82,538
dk/dk_2024,"617,987",99.9,132,58,573
es_cat/es_cat_2024,"723,039",100.0,155,73,654
si/si_2024,"809,044",100.0,111,58,741
fi/fi_2024,"998,336",100.0,119,63,928
ie/ie_2024,"1,119,949",100.0,88,30,"1,061"


In [9]:
fields  = load_columns(build_file_url(COLLECTION,'fields'),['hcat:code','hcat:name'])
crops   = fields.value_counts(['hcat:code','hcat:name']).rename('nfields').reset_index()
crops['pct'] = 100*crops.nfields/len(fields)
kept    = crops[crops.nfields>=MINFIELDS]
headers = {'hcat:code':'HCAT Code','hcat:name':'HCAT Class','nfields':'Fields','pct':'Fields (%)'}

display(kept.rename(columns=headers).style.format({'Fields':'{:,}','Fields (%)':'{:.2f}'}).hide(axis='index'))

HCAT Code,HCAT Class,Fields,Fields (%)
3302000000,pasture_meadow_grassland_grass,"74,806",44.93
3301010101,winter_common_soft_wheat,"14,180",8.52
3301090300,legumes_harvested_green,"12,489",7.50
3301010500,oats,"7,680",4.61
3301010402,spring_barley,"7,106",4.27
3301090303,clover,"6,788",4.08
3301020600,peas,"6,159",3.70
3301060401,winter_rapeseed_rape,"4,624",2.78
3301010102,spring_common_soft_wheat,"4,566",2.74
3301030000,potatoes,"2,883",1.73
